# Student Performance Data Explorer
**Day 7 – Mini Project | AI/ML Internship, Sumerix Global**

**Problem:** Institutions collect student performance data but often lack insight into trends and learning outcomes. This notebook cleans the data, explores it, and summarizes key findings.

**Workflow:** Load → Clean → Analyze → Visualize → Insights

## 1. Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
os.makedirs('../images', exist_ok=True)

df = pd.read_csv('../data/student_data.csv')
print('Shape:', df.shape)
df.head(10)

## 2. Dataset Overview

In [ ]:
df.info()

In [ ]:
print('Columns:', list(df.columns))
df.describe(include='all').T

## 3. Data Cleaning

### 3.1 Standardize column names

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(' ', '_').str.replace('/', '_'))
print(list(df.columns))

### 3.2 Missing value analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'percent': missing_pct})

In [ ]:
# Fill numeric columns with the median, categorical columns with the mode
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].mode()[0])

print('Missing values left:', df.isnull().sum().sum())

### 3.3 Remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows. Rows now: {len(df)}')

In [ ]:
# Save the cleaned dataset (needed for Task 2)
df.to_csv('../data/student_data_cleaned.csv', index=False)

## 4. Statistical Summary

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
summary = df[num_cols].agg(['mean', 'median', 'std', 'min', 'max']).T
summary['mode'] = [df[c].mode()[0] for c in num_cols]
summary.round(2)

## 5. Visualizations

### 5.1 Histograms of all numeric columns

In [ ]:
df[num_cols].hist(figsize=(12, 8), bins=15, edgecolor='black')
plt.suptitle('Distribution of Numeric Columns', y=1.02)
plt.tight_layout()
plt.savefig('../images/mini_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** _write what you notice (skew, typical range, any odd values)._

### 5.2 Box plots (outlier detection)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[num_cols], orient='h')
plt.title('Box Plots of Numeric Columns')
plt.xlabel('Value')
plt.ylabel('Column')
plt.savefig('../images/mini_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** _which columns have outliers?_

### 5.3 Correlation heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap')
plt.savefig('../images/mini_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** _which pairs are most strongly correlated?_

### 5.4 Scatter plot of two strongest-related columns

In [ ]:
corr = df[num_cols].corr().abs()
np.fill_diagonal(corr.values, 0)
x, y = corr.stack().idxmax()
print('Most correlated pair:', x, 'and', y)

plt.figure(figsize=(7, 5))
sns.regplot(data=df, x=x, y=y)
plt.title(f'{x} vs {y}')
plt.xlabel(x)
plt.ylabel(y)
plt.savefig('../images/mini_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Performance by category (only runs if the dataset has categorical columns)

In [ ]:
target = num_cols[0]  # change to e.g. 'math_score' if you prefer
for c in cat_cols[:2]:
    if df[c].nunique() <= 10:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=c, y=target)
        plt.title(f'{target} by {c}')
        plt.xlabel(c)
        plt.ylabel(target)
        plt.xticks(rotation=30)
        plt.savefig(f'../images/mini_{c}_vs_{target}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 6. Key Insights
Replace these with findings from your own charts:
1. _Average score and spread of the main subject_
2. _The strongest correlation found and what it suggests_
3. _Outliers or unusual patterns_
4. _Differences between groups (gender, parental education, etc.) if present_

## 7. Conclusion & Next Steps
The cleaned dataset (`data/student_data_cleaned.csv`) is ready for Task 2: preprocessing, feature engineering, and baseline models.